# Problem and data understanding

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from src.data_loader import load_partition, predictors, fingerprints
def report(name):
    display(Markdown((ROOT / 'reports' / name).read_text(encoding='utf-8')))
def figure(name):
    display(Image(filename=str(ROOT / 'figures' / name)))


Define binary flow classification, its decision context and research targets. Data labels are kept out of predictors.

In [2]:
report('problem_statement.md')

# Problem statement

## Security problem and business value
Security Operations Centers (SOCs) face more network telemetry than analysts can inspect manually. A supervised model can prioritize flows for investigation, but missed attacks expose systems to harm and false alerts consume analyst time. The proposed system provides decision support to analysts who can combine model output with endpoint, identity and asset context.

## Research question and task
Can supervised machine-learning models distinguish malicious from benign network flows while maintaining an operationally acceptable balance of missed attacks and false alerts? This is binary classification: `label=0` denotes benign and `label=1` denotes attack. It is not an experiment demonstrating detection of genuinely novel attacks, nor does binary classification identify an attack family. `attack_cat` remains available only for explanation and subgroup analysis.

## Measurable success criteria
Research targets are F1 >= 0.90 and ROC-AUC >= 0.95, where achievable. Report attack recall, precision, average precision, FPR and FNR regardless of target attainment. A provisional validation operating constraint asks for recall >= 0.95 and FPR <= 0.05; maximize F1 among feasible thresholds. If no threshold is feasible, explicitly report failure and use maximum validation F1 as an exploratory operating point. These research constraints need stakeholder approval and local workload validation before operational use.

## Lifecycle and constraints
Frame the decision; acquire and validate data; isolate evaluation data; explore training data; fit preprocessing inside cross-validation; tune and compare models; freeze a validation threshold; evaluate once; explain decisions; audit errors; deploy a local prototype; propose monitoring and retraining. Accuracy alone can hide missed attacks and alert burden. Feature extraction must be available at decision time. Flow completion introduces latency; aggregate connection counters require historical context. No packet-capture or live blocking integration is included.

## Risks
False negatives can delay investigation. False positives can cause alert fatigue. Scores require calibration before probability-sensitive decisions. Explainability supports inspection but does not establish causality. Historical laboratory traffic differs from current deployment environments. Concept drift, changing service distributions and attacker adaptation can reduce performance; test on recent local telemetry and retain human review.


In [3]:
train_raw=load_partition('training')
test_raw=load_partition('testing')
assert len(train_raw)==175341 and len(test_raw)==82332
display(pd.read_csv(ROOT/'reports/dataset_summary.csv'))
display(train_raw.dtypes.to_frame('type'))

,partition,rows,columns,missing_cells,exact_duplicates,predictor_duplicates,benign,attack
0,Published training,175341,45,0,0,74301,56000,119341
1,Published test,82332,45,0,0,28386,37000,45332
2,Development train,80648,45,0,0,0,41328,39320


,type
id,int64
dur,float64
proto,object
service,object
state,object
spkts,int64
dpkts,int64
sbytes,int64
dbytes,int64
rate,float64


In [4]:
display(train_raw.isna().sum().to_frame('missing'))
display(train_raw.label.value_counts().to_frame('count'))
display(train_raw.attack_cat.value_counts().to_frame('count'))
display(pd.read_csv(ROOT/'reports/data_dictionary.csv'))

,missing
id,0
dur,0
proto,0
service,0
state,0
spkts,0
dpkts,0
sbytes,0
dbytes,0
rate,0


,count
label,
1,119341
0,56000


,count
attack_cat,
Normal,56000
Generic,40000
Exploits,33393
Fuzzers,18184
DoS,12264
Reconnaissance,10491
Analysis,2000
Backdoor,1746
Shellcode,1133


,feature_name,data_type,category,description,role,preprocessing,possible_security_meaning
0,id,int64,annotation,Partition-local record identifier; no predicti...,excluded identifier,Excluded from X,Never use as model input.
1,dur,float64,flow statistic,Record total duration,predictor,Finite values; train-fitted median and standar...,Describes traffic behavior; association alone ...
2,proto,object,categorical,Transaction protocol,predictor,"Train-fitted mode, rare-category one-hot encoding",Describes traffic behavior; association alone ...
3,service,object,categorical,"http, ftp, smtp, ssh, dns, ftp-data ,irc and ...",predictor,"Train-fitted mode, rare-category one-hot encoding",Describes traffic behavior; association alone ...
4,state,object,categorical,Indicates to the state and its dependent proto...,predictor,"Train-fitted mode, rare-category one-hot encoding",Describes traffic behavior; association alone ...
5,spkts,int64,flow statistic,Source to destination packet count,predictor,Finite values; train-fitted median and standar...,Describes traffic behavior; association alone ...
6,dpkts,int64,flow statistic,Destination to source packet count,predictor,Finite values; train-fitted median and standar...,Describes traffic behavior; association alone ...
7,sbytes,int64,flow statistic,Source to destination transaction bytes,predictor,Finite values; train-fitted median and standar...,Describes traffic behavior; association alone ...
8,dbytes,int64,flow statistic,Destination to source transaction bytes,predictor,Finite values; train-fitted median and standar...,Describes traffic behavior; association alone ...
9,rate,float64,flow statistic,Flow packet rate supplied by the dataset; pres...,predictor,Finite values; train-fitted median and standar...,Describes traffic behavior; association alone ...


Interpretation: raw partition totals differ from the cleaned modeling populations. Predictor duplicates ignore the partition-local ID. Provenance normalizes a filename reversal in the mirror; official checksum equivalence remains unverified.

In [5]:
report('dataset_documentation.md')

# Dataset documentation

## Source and provenance
Use the [UNSW-NB15 dataset](https://research.unsw.edu.au/projects/unsw-nb15-dataset), obtained through the public mirror because the university download redirected to sign-in. The original project describes traffic collected in a controlled cyber range, with benign activity and generated attack behavior. Its public partitions contain 175,341 training and 82,332 testing records.

**Mirror naming discrepancy:** the mirror file named `UNSW_NB15_testing-set.csv` contains the 175,341-row partition, while its `training` file contains 82,332 rows. The downloader maps by these published row counts and records source filenames, URLs and SHA-256 checksums. This verifies internal consistency, not byte-level authenticity against the inaccessible official download. Retain this limitation in comparisons with published work.

## Observed summary

| partition          |   rows |   columns |   missing_cells |   exact_duplicates |   predictor_duplicates |   benign |   attack |
|:-------------------|-------:|----------:|----------------:|-------------------:|-----------------------:|---------:|---------:|
| Published training | 175341 |        45 |               0 |                  0 |                  74301 |    56000 |   119341 |
| Published test     |  82332 |        45 |               0 |                  0 |                  28386 |    37000 |    45332 |
| Development train  |  80648 |        45 |               0 |                  0 |                      0 |    41328 |    39320 |

The partition CSV schema has 45 columns: 42 original predictors, an identifier, the binary label, and attack category. Predictor categories are protocol, service and state; the remaining original predictors are numerical. These partition features differ from the full raw-flow dictionary. Aliases are mapped in `src/reporting.py` and the delivered dictionary includes six engineered features.

## Quality controls
See `missing_values.csv`, `feature_types.csv`, `numeric_summary.csv`, `negative_values.csv`, and `attack_distribution.csv` for computed per-column evidence. A literal service `-` is retained as a legitimate unspecified service, not declared missing. Infinite numeric values become missing. No benign/attack label is imputed. Category labels are stripped of surrounding whitespace. Zero packet/duration denominators produce missing engineered ratios rather than infinity.

## Duplicate policy and split sizes

```json
{
  "raw_train": 175341,
  "raw_test": 82332,
  "clean_development": 100811,
  "training": 80648,
  "validation": 20163,
  "primary_test": 52738,
  "removed_development": 74530,
  "conflicting_development_signatures": 229,
  "removed_test_duplicates_or_conflicts": 28392,
  "conflicting_test_signatures": 6,
  "test_overlap_removed": 1202,
  "seed": 42
}
```

Predictor-identical development rows are deduplicated irrespective of ID. Ambiguous signatures with conflicting binary labels are removed and counted. Development data are split 80/20 with stratification and seed 42. All preprocessing and CV use only the training side. For the primary test analysis, duplicates/conflicting signatures are removed within the published test and signatures seen in development are excluded. This changes the evaluation population; `published_test_secondary.json` separately retains the entire published test population for transparency. No test outcomes determine the model or threshold. Exact hashes do not catch near-duplicates or shared capture sessions, and no trustworthy session/time key is available in these partition CSVs.

## Data dictionary and terms
`data_dictionary.csv` records feature name, observed type, feature category, description, modeling role, preprocessing and possible security meaning. It uses the mirrored source dictionary with explicit aliases. Do not assume exact equivalence between these fields and arbitrary NetFlow exports.
The dataset remains under its authors' terms: academic research is permitted; commercial use requires agreement with the authors. The repository code license does not relicense the dataset.
